# Preprocessing

## Análisis y preprocesado de los datos

Iniciamos leyendo los datos, que contienen varios errores a arreglar durante el preprocesado

In [43]:
import numpy as np
import pandas as pd
from load import read_data

df = read_data(filepath="../data/Autism-Adult-Data.csv")

#### Error en representación de valores perdidos

Los valores perdidos en este dataset están representados como ?, y nos sirve que estén representados como NaN, ya que sino las columnas quedarán como dtype='object' y necesitamos que sean numéricas.

Demostración del error:

In [44]:
print(np.unique(df.age)) # Debería ser numérica
print(np.unique(df.result)) # Debería ser numérica (en este caso no hace falta arreglarla 
                            # porque no hay valores perdidos)

['17' '18' '19' '20' '21' '22' '23' '24' '25' '26' '27' '28' '29' '30'
 '31' '32' '33' '34' '35' '36' '37' '38' '383' '39' '40' '41' '42' '43'
 '44' '45' '46' '47' '48' '49' '50' '51' '52' '53' '54' '55' '56' '58'
 '59' '60' '61' '64' '?']
[ 0  1  2  3  4  5  6  7  8  9 10]


In [45]:
idxs_numeric_cols = [11] #[11, 18]

for idx in idxs_numeric_cols:
    # errors coerce cambia no numericos ('?') a NaN
    df.iloc[:, idx] = pd.to_numeric(df.iloc[:, idx], errors='coerce')

Confirmación de que se realizó la operación correctamente:

In [46]:
print(np.unique(df.age)) # Debería ser numérica

[17.0 18.0 19.0 20.0 21.0 22.0 23.0 24.0 25.0 26.0 27.0 nan 27.0 28.0 29.0
 30.0 nan 30.0 31.0 32.0 33.0 34.0 35.0 36.0 37.0 38.0 39.0 40.0 41.0 42.0
 43.0 44.0 45.0 46.0 47.0 48.0 49.0 50.0 51.0 52.0 53.0 54.0 55.0 56.0
 58.0 59.0 60.0 61.0 64.0 383.0]


### Errores de nombrado y columna innecesaria

Las columnas ID y age_desc no aportan información, ya que ID contiene valores diferentes incrementales por cada uno de los datos, y age_desc contiene un único valor diferente: '18 and more'

In [47]:
print(f"Unique vals in age_desc: {np.unique(df.age_desc)}")

print(f"# of rows in dataframe: {df.shape[0]}")
print(f"# of unique vals in ID: {len(np.unique(df.id))}")

df = df.drop(columns=["id", "age_desc"])

Unique vals in age_desc: ['18 and more']
# of rows in dataframe: 704
# of unique vals in ID: 704


### Columnas listas para valores faltantes, outliers e inconsistencias

Vamos a hacer una breve muestra de como quedaron los datos

In [48]:
def describe_data(df: pd.DataFrame) -> None:
    columns = df.columns
    describe = df.describe()

    for c in columns:
        print(f"\n\nColumna: {c}")
        print(f"DataType: {df.loc[:,c].dtype}")
        print(f"Unique Vals: {np.unique(df.loc[:,c])}")
        if c in describe:
            print(f"Description:\n{describe.loc[:,c]}")
    
describe_data(df)



Columna: A1_Score
DataType: int64
Unique Vals: [0 1]
Description:
count    704.000000
mean       0.721591
std        0.448535
min        0.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: A1_Score, dtype: float64


Columna: A2_Score
DataType: int64
Unique Vals: [0 1]
Description:
count    704.000000
mean       0.453125
std        0.498152
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000
Name: A2_Score, dtype: float64


Columna: A3_Score
DataType: int64
Unique Vals: [0 1]
Description:
count    704.000000
mean       0.457386
std        0.498535
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000
Name: A3_Score, dtype: float64


Columna: A4_Score
DataType: int64
Unique Vals: [0 1]
Description:
count    704.000000
mean       0.495739
std        0.500337
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.00

### Categorico a numérico

Tenemos varias columnas binarias cuyos valores son {'no', 'yes'}, {'f', 'm'}, ...

Estas columnas las reemplazaremos por columnas enteras con estos reemplazamientos:

    no, f = 0
    yes, m = 1

In [ ]:
idxs_categoric_binary_columns = [11, 13, 14, 16, 19]

